### Dataset

In [1]:
from chronos import BaseChronosPipeline

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-t5-tiny"
)

c:\Users\Federico\DesktopW\patchAliasing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
import pandas as pd
from chronos import BaseChronosPipeline

pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-tiny")

# Load historical data
db_url = "dataset/Dataset-SolarTechLabEngineered.csv"
df = pd.read_csv(db_url, delimiter=",")

In [8]:
df.columns

Index(['Date', 'Power_W', 'Temp_Celsius', 'Irradiance_horizontalPlane',
       'Irradiance_inclinedPlane', 'Wind_speed', 'Wind_direction'],
      dtype='str')

In [13]:
df['item_id'] = 'power_series'  # Add an item_id column for the time series identifier


df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["item_id", "Date"]).reset_index(drop=True)

pred_df = pipeline.predict_df(
    df,
    prediction_length=64,
    quantile_levels=[0.1, 0.5, 0.9],
    id_column="item_id",
    timestamp_column="Date",
    target="Power_W",
    freq="min",
)

ValueError: Could not infer frequency for series power_series

In [ ]:
pred_df.head()

,item_id,Month,target_name,predictions,0.1,0.5,0.9
0,T1,1961-01-01,#Passengers,442.025238,418.598541,443.874756,472.644379
1,T1,1961-02-01,#Passengers,445.313141,414.899384,447.984711,468.534424
2,T1,1961-03-01,#Passengers,462.472229,429.489807,462.369507,492.166504
3,T1,1961-04-01,#Passengers,479.733978,445.724121,478.809296,507.989777
4,T1,1961-05-01,#Passengers,515.387817,458.259430,518.881226,560.391724


In [15]:
from chronos import BaseChronosPipeline
import torch

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-t5-tiny",
    device_map="cpu",
)

context = torch.tensor([1.2, 1.5, 1.7, 1.4, 1.8])

token_ids, attention_mask, tokenizer_state = pipeline.tokenizer.context_input_transform(context.unsqueeze(0))


In [16]:
embed_layer = pipeline.model.model.encoder.embed_tokens

In [17]:
embeddings = embed_layer(token_ids)

print(embeddings.shape)

torch.Size([1, 6, 256])


In [19]:
outputs = pipeline.model.model.encoder(
    input_ids=token_ids,
    attention_mask=attention_mask,
    output_hidden_states=True,
    return_dict=True,
)

hidden_states = outputs.hidden_states
print(len(hidden_states))

5


In [24]:
from chronos import BaseChronosPipeline

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-bolt-tiny",
    device_map="cpu",
)

model = pipeline.model

print(type(model))

print("\n================ MODULES ================\n")

for name, module in model.named_modules():
    print(name, " ---> ", type(module))

<class 'chronos.chronos_bolt.ChronosBoltModelForForecasting'>

================ MODULES ================

  --->  <class 'chronos.chronos_bolt.ChronosBoltModelForForecasting'>
shared  --->  <class 'torch.nn.modules.sparse.Embedding'>
input_patch_embedding  --->  <class 'chronos.chronos_bolt.ResidualBlock'>
input_patch_embedding.dropout  --->  <class 'torch.nn.modules.dropout.Dropout'>
input_patch_embedding.hidden_layer  --->  <class 'torch.nn.modules.linear.Linear'>
input_patch_embedding.act  --->  <class 'torch.nn.modules.activation.ReLU'>
input_patch_embedding.output_layer  --->  <class 'torch.nn.modules.linear.Linear'>
input_patch_embedding.residual_layer  --->  <class 'torch.nn.modules.linear.Linear'>
patch  --->  <class 'chronos.chronos_bolt.Patch'>
instance_norm  --->  <class 'chronos.chronos_bolt.InstanceNorm'>
encoder  --->  <class 'transformers.models.t5.modeling_t5.T5Stack'>
encoder.block  --->  <class 'torch.nn.modules.container.ModuleList'>
encoder.block.0  --->  <class 'tr

In [28]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from chronos import BaseChronosPipeline
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# LOAD
# ============================================================

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-bolt-tiny",
    device_map="cpu",
)

model = pipeline.model


# ============================================================
# STORAGE
# ============================================================

captured = {
    "patches": None,
    "patch_embeddings": None,
    "encoder_inputs": None,
}


# ============================================================
# HOOKS
# ============================================================

def patch_hook(module, inputs, outputs):

    captured["patches"] = outputs.detach().cpu()


def embedding_hook(module, inputs, outputs):

    captured["patch_embeddings"] = outputs.detach().cpu()


def encoder_hook(module, inputs, outputs):

    for x in inputs:

        if torch.is_tensor(x):

            if x.ndim == 3:

                captured["encoder_inputs"] = x.detach().cpu()


# ============================================================
# REGISTER
# ============================================================

h1 = model.patch.register_forward_hook(patch_hook)
# register_forward_hook registers a function to be called every time the module executes a forward pass. The hook function receives the module, its inputs, and its outputs as arguments.
h2 = model.input_patch_embedding.register_forward_hook(
    embedding_hook
)

h3 = model.encoder.register_forward_hook(
    encoder_hook
)
# hook is a handle that can be used to remove the hook later

# ============================================================
# INPUT
# ============================================================

series = torch.tensor([
    1.0, 1.2, 1.4, 1.1,
    0.9, 1.3, 1.5, 1.7,
    1.8, 1.6, 1.4, 1.2,
    1.1, 1.0, 0.8, 0.7,
    0.9, 1.1, 1.3, 1.5,
    1.7, 1.9, 2.0, 1.8,
    1.6, 1.4, 1.2, 1.0,
]).float()


# ============================================================
# RUN THROUGH PIPELINE
# ============================================================

with torch.no_grad():

    forecast = pipeline.predict(
        series,
        prediction_length=8,
    )


# ============================================================
# REMOVE HOOKS
# ============================================================

h1.remove()
h2.remove()
h3.remove()


# ============================================================
# RESULTS
# ============================================================

patches = captured["patches"]
emb = captured["patch_embeddings"]
enc = captured["encoder_inputs"]


print("\n================================================")
print("PATCHES")
print("================================================")

print(patches.shape)

print("\nFirst patch:")
print(patches[0, 0])
print("\nSecond patch:")
print(patches[0, 1])


print("\n================================================")
print("PATCH EMBEDDINGS")
print("================================================")

print(emb.shape)

batch, num_patches, d_model = emb.shape

print("\nNum patches:", num_patches)
print("Embedding dim:", d_model)

print("\nFirst embedding:")
print(emb[0, 0])


print("\n================================================")
print("ENCODER INPUT")
print("================================================")

print(enc.shape)


# ============================================================
# CHECK EQUALITY
# ============================================================

print("\n================================================")
print("ARE PATCH EMBEDDINGS == ENCODER INPUT?")
print("================================================")

diff = torch.abs(emb - enc).mean()

print("Mean absolute difference:", diff.item())


# ============================================================
# NORMS
# ============================================================

norms = torch.norm(emb[0], dim=-1)

plt.figure(figsize=(10, 4))
plt.plot(norms.numpy())
plt.title("Patch Embedding Norms")
plt.xlabel("Patch")
plt.ylabel("Norm")
plt.grid()
plt.show()


# ============================================================
# COSINE SIMILARITY
# ============================================================

X = emb[0].numpy()

sim = cosine_similarity(X)

plt.figure(figsize=(8, 8))
plt.imshow(sim)
plt.colorbar()
plt.title("Patch Embedding Cosine Similarity")
plt.show()


# ============================================================
# PCA
# ============================================================

pca = PCA(n_components=2)

Y = pca.fit_transform(X)

plt.figure(figsize=(8, 6))

for i in range(num_patches):

    plt.scatter(Y[i, 0], Y[i, 1])
    plt.text(Y[i, 0], Y[i, 1], str(i))

plt.title("Patch Embedding PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid()
plt.show()


# ============================================================
# TEMPORAL SIMILARITY
# ============================================================

adjacent = []

for i in range(num_patches - 1):

    a = X[i]
    b = X[i + 1]

    cos = np.dot(a, b) / (
        np.linalg.norm(a)
        * np.linalg.norm(b)
    )

    adjacent.append(cos)

plt.figure(figsize=(10, 4))
plt.plot(adjacent)
plt.title("Adjacent Patch Similarity")
plt.xlabel("Patch")
plt.ylabel("Cosine similarity")
plt.grid()
plt.show()


# ============================================================
# FORECAST
# ============================================================

print("\n================================================")
print("FORECAST")
print("================================================")

print(forecast)


PATCHES
torch.Size([1, 2, 16])

First patch:
tensor([nan, nan, nan, nan, 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

Second patch:
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

PATCH EMBEDDINGS
torch.Size([1, 2, 256])

Num patches: 2
Embedding dim: 256

First embedding:
tensor([-3.1560e-03, -8.6343e-02,  8.6576e-02, -6.0975e-02, -7.7855e-03,
         6.2622e-02, -1.9718e-02, -8.2131e-03,  2.7897e-02, -3.3527e-02,
        -7.0937e-02, -2.1748e-02,  2.8534e-01, -9.7734e-02, -3.4970e-02,
         3.2531e-03, -4.0624e-02, -5.3894e-02, -9.6966e-02,  4.5243e-02,
        -1.5248e-01, -1.2590e-02, -2.5783e-02, -4.2249e-02, -2.6259e-02,
        -4.4527e-02,  5.6483e-03, -2.5348e-02, -5.2015e-02,  9.9663e-03,
         8.6720e-02,  4.3890e-03,  7.0396e-02, -1.0269e-02, -6.1225e-02,
        -5.1157e-02, -5.5313e-02, -1.3915e-02, -6.9297e-02, -6.4343e-03,
         1.0005e-01,  5.0827e-02,  3.3206e-02,  3.3883e-02,  3.5720e-02,
        -9.3577e-03,  1.0859e-01, -1.

AttributeError: 'NoneType' object has no attribute 'shape'